In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, current_timestamp
from datetime import datetime, date

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("bronze.boe_base_rate")

print(f"Rows read from bronze: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
df_bronze = spark.table("bronze.boe_base_rate")
df_bronze = df_bronze.alias("df_bronze")
print(df_bronze.columns)

In [0]:
df_silver = df_bronze \
    .withColumn("report_date", to_date(col("DATE"), "dd MMM yyyy")) \
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM")) \
    .drop("DATE")

# Sanity check - confirm date cast and year_month derivation
df_silver.select("report_date", "year_month", "base_rate_pct").show(5)

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.boe_base_rate")
)

print(f"Written {df_silver.count()} rows to silver.boe_base_rate")

In [0]:
# Sanity check - confirm rate cycle looks historically accurate
spark.sql("""
    SELECT report_date, year_month, base_rate_pct
    FROM silver.boe_base_rate
    ORDER BY report_date DESC
    LIMIT 5
""").show()

In [0]:
# Silver transformation complete
# Source: bronze.boe_base_rate
# Rows in: 268 | Rows written: 268
# Transformations applied:
#   - Cast DATE from string (dd MMM yyyy) to date type, renamed to report_date
#   - Derived year_month join key (yyyy-MM string)
#   - Dropped original DATE string column
# Note: 'date' is a reserved word in this Spark version - use report_date consistently
# Note: date_format requires .cast("timestamp") before report_date in this Spark version
# Table: silver.boe_base_rate

print("silver_03_boe_base_rate complete.")